In [1]:
import joblib
import os
import numpy as np
import pandas as pd
import json
from sklearn.metrics import roc_auc_score
class EnsemblePredictionAPI:
    def __init__(self, folder_path='models/'):
      
        self.folder_path = folder_path
        self.models, self.model_names, self.n_folds = self.load_all_models()

    def load_all_models(self):
        
        models = []
        model_names = set()
        
        for file_name in os.listdir(self.folder_path):
            if file_name.endswith('.joblib'):
                parts = file_name.split('_fold_')
                model_name = parts[0]
                fold_idx = int(parts[1].replace('.joblib', ''))
    
                while len(models) <= fold_idx:
                    models.append({})
    
                models[fold_idx][model_name] = joblib.load(os.path.join(self.folder_path, file_name))
                model_names.add(model_name)
        
        n_folds = len(models)
        return models, list(model_names), n_folds

    
    def ensemble_predict(self, test_data, strategy='average'):
        
        predictions = np.zeros((test_data.shape[0], len(self.model_names), self.n_folds))

        for i in range(self.n_folds):
            for j, model_name in enumerate(self.model_names):
                predictions[:, j, i] = self.models[i][model_name].predict_proba(test_data)[:, 1]

        if strategy == 'average':
            avg_predictions = np.mean(np.mean(predictions, axis=2), axis=1)
        elif strategy == 'median':
            avg_predictions = np.median(np.mean(predictions, axis=2), axis=1)
        else:
            raise ValueError(f"Ты шо ввел?: {strategy}")
        
        result = {"predictions": avg_predictions.tolist()}
        return result

    def process(self, test_data):
        self.cols_to_concat = test_data[['contract_id', 'report_date']]
        return test_data.fillna(0).drop(['report_date', 'contract_date', 'contract_id'], axis=1)

    def predict(self, test_data, strategy='average'):
 
        test_data = self.process(test_data)
        
        predictions = self.ensemble_predict(test_data, strategy)
        result = pd.concat([self.cols_to_concat, pd.Series(predictions['predictions'], name='score')], axis=1)
        
        return result, json.loads(result.groupby('contract_id').apply(lambda x: x.to_json(orient='values')).to_json())
    

In [2]:
test_cv = pd.read_csv('../data/test1_X.csv').fillna(0)

In [3]:
api = EnsemblePredictionAPI()


In [4]:
sumbit = api.predict(test_cv, strategy='median')

In [5]:
sumbit.to_csv('../submissions/last_2.csv', index=False)

In [60]:
def advanced_momentum_adjustment(df, momentum_factor=0.05):
    
    def apply_momentum(group):
        group = group.copy()
        
        group['prev_score'] = group['score'].shift(1)
        group['prev_report_date'] = group['report_date'].shift(1)
        
        def adjust_score(row):
            if pd.isna(row['prev_score']):
                return row['score']
            elif row['score'] < row['prev_score']:
                return max(row['score'] - momentum_factor, 0)
            elif row['score'] > row['prev_score'] and row['score'] > 0.5:
                return min(row['score'] + momentum_factor, 1)
            else:
                return row['score']
        
        group['adjusted_score'] = group.apply(adjust_score, axis=1)
        
        return group

    df_adjusted = df.groupby('contract_id', group_keys=False).apply(apply_momentum)
    
    df_adjusted = df_adjusted.drop(columns=['prev_score', 'prev_report_date'])
    
    return df_adjusted


In [51]:
sumbit

,contract_id,report_date,score
0,3029,2023-07-30,0.033574
1,4350,2023-07-30,0.042397
2,1095,2023-07-30,0.028397
3,2634,2023-07-30,0.025620
4,6535,2023-07-30,0.028918
...,...,...,...
7327,4740,2023-09-24,0.026745
7328,2689,2023-09-24,0.132912
7329,7135,2023-09-24,0.136576
7330,4164,2023-09-24,0.062760
